In [21]:
import os
import random
import h5py
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio.transforms as T
from diffusers import DiffusionPipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# EEG Projector -> latent grid
# -----------------------------
class EEGProjector(nn.Module):
    """
    Maps EEG (B, Fy, Sy) -> (B, Ceeg, Fz, Sz)
    """
    def __init__(self, Fy, Fz, Sz, Ceeg=8):
        super().__init__()
        self.Fz = Fz
        self.Sz = Sz
        self.Ceeg = Ceeg

        self.proj = nn.Sequential(
            nn.Conv1d(Fy, 256, kernel_size=5, stride=5),
            nn.ReLU(inplace=True),
            nn.Conv1d(256, 512, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv1d(512, 1024, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv1d(1024, 2048, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
        )
        # reduce channels 2048 → Ceeg
        self.reduce = nn.Conv1d(2048, Ceeg, kernel_size=1)

    def forward(self, eeg):  # eeg: (B, Fy, Sy)
        x = self.proj(eeg)            # (B, 2048, ~Sy/40)
        x = self.reduce(x)            # (B, Ceeg, ~Sy/40)
        x = F.adaptive_avg_pool1d(x, self.Fz * self.Sz)  # (B, Ceeg, Fz*Sz)
        x = x.view(eeg.size(0), self.Ceeg, self.Fz, self.Sz)
        return x

# -----------------------------
# Load AudioLDM2 pipeline
# -----------------------------
pipeline = DiffusionPipeline.from_pretrained(
    "cvssp/audioldm2-music",
    torch_dtype=torch.float16,
    trust_remote_code=True
).to(device)

pipeline.unet.train()  # we will fine-tune UNet

# Prepare prompt embeddings once
prompt = "Pop music"
with torch.no_grad():
    prompt_embeds, attention_mask, _ = pipeline.encode_prompt(
        prompt=prompt,
        device=device,
        num_waveforms_per_prompt=1,
        do_classifier_free_guidance=False
    )

# -----------------------------
# Data prep: EEG ↔ audio latents
# -----------------------------
data_path = 'data'
data = h5py.File(os.path.join(data_path, 'madeeg_preprocessed.hdf5'), 'r')
with open(os.path.join(data_path, 'madeeg_preprocessed.yaml'), 'r') as stream:
    metadata = yaml.load(stream, Loader=yaml.FullLoader)

fs = 256
num_sec = 1
eeg_X = []
z_latents = []

latent_C = latent_F = latent_T = None

subjects = list(data.keys())
for sbj in subjects[:1]:
    stimuli = list(metadata[sbj].keys())
    for stim in stimuli[:2]:
        response = data[sbj][stim]['response']  # (Fy, time)
        Fy = response.shape[0]

        # trial averaging across 4 repetitions
        interval_length = response.shape[1] // 4
        mean_eeg = sum(response[:, i*interval_length:(i+1)*interval_length] for i in range(4)) / 4.0

        stimulus = data[sbj][stim]['stimulus']   # (2, samples)
        sfreq = metadata[sbj][stim]['wav_info']['sfreq']
        pipeline.feature_extractor.sampling_rate = int(sfreq)

        mel_transform = T.MelSpectrogram(sample_rate=int(sfreq), n_mels=128)

        seg = 0
        while (seg + num_sec) * fs < mean_eeg.shape[1]:
            eeg_tensor = torch.from_numpy(mean_eeg[:, seg*fs:(seg+1)*fs]).float()  # (Fy, Sy)

            ch1 = torch.from_numpy(stimulus[0, :]).float()
            ch2 = torch.from_numpy(stimulus[1, :]).float()
            mix = (ch1 + ch2)[seg*int(sfreq):(seg+1)*int(sfreq)]

            with torch.no_grad():
                mel = mel_transform(mix.unsqueeze(0)).to(device, dtype=torch.float16)  # (1, 128, T)
                latent_dist = pipeline.vae.tiled_encode(mel.unsqueeze(0))
                x0 = latent_dist.latent_dist.sample()  # (1, C, F, T)

            if latent_F is None:
                latent_C, latent_F, latent_T = x0.shape[1:]

            eeg_X.append(eeg_tensor)
            z_latents.append(x0.squeeze(0).cpu())
            seg += 1
    print(f"subject {sbj} data gathered")

print(f"{len(eeg_X)} samples gathered")
print(f"Latent shape: C={latent_C}, F={latent_F}, T={latent_T}")

# -----------------------------
# EEG-conditioned UNet
# -----------------------------
class EEGConditionedUNet(nn.Module):
    def __init__(self, base_model, Fy, Fz, Sz, Ceeg=8):
        super().__init__()
        self.unet = base_model.unet
        self.projector = EEGProjector(Fy=Fy, Fz=Fz, Sz=Sz, Ceeg=Ceeg)
        # reduce EEG channels → UNet channels
        self.proj_conv = nn.Conv2d(Ceeg, self.unet.config.in_channels, kernel_size=1)
        # project text embeddings
        self.text_proj = nn.Linear(base_model.text_encoder.config.hidden_size, 1024)

    def forward(self, x_t, eeg_map, t, prompt_embeds):
        # EEG → latent
        eeg_proj = self.projector(eeg_map)         # (B, Ceeg, Fz, Sz)
        eeg_proj = self.proj_conv(eeg_proj)        # (B, Cz, Fz, Sz)
        x_in = x_t + eeg_proj                       # residual conditioning

        # Text embeddings
        prompt_proj = self.text_proj(prompt_embeds)  # (B, seq_len, 1024)

        return self.unet(
            sample=x_in,
            timestep=t,
            encoder_hidden_states=prompt_proj
        ).sample

# -----------------------------
# Instantiate model + optimizer
# -----------------------------
Fy = eeg_X[0].shape[0]
Fz, Sz = latent_F, latent_T
model = EEGConditionedUNet(pipeline, Fy, Fz, Sz).to(device, dtype=torch.float16)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# -----------------------------
# Training loop
# -----------------------------
pairs = list(zip(eeg_X, z_latents))
num_epochs = 30
batch_size = 2

def collate(batch):
    eeg_batch = torch.stack([b[0] for b in batch])   # (B, Fy, Sy)
    x0_batch  = torch.stack([b[1] for b in batch])   # (B, Cz, Fz, Sz)
    return eeg_batch, x0_batch

for epoch in range(num_epochs):
    random.shuffle(pairs)
    running = 0.0

    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        if len(batch) < batch_size:
            continue

        eeg_batch, x0_batch = collate(batch)
        eeg_batch = eeg_batch.to(device, dtype=torch.float16)
        x0_batch  = x0_batch.to(device, dtype=torch.float16)
        B = x0_batch.size(0)

        # Diffusion timestep
        t = torch.randint(
            low=0,
            high=pipeline.scheduler.num_train_timesteps,
            size=(B,),
            device=device,
            dtype=torch.long
        )
        noise = torch.randn_like(x0_batch)
        x_t = pipeline.scheduler.add_noise(x0_batch, noise, t)

        # Expand prompt embeddings if needed
        pe = prompt_embeds
        if pe.size(0) != B:
            pe = pe.repeat(B, 1, 1)

        optimizer.zero_grad()
        eps_pred = model(x_t=x_t, eeg_map=eeg_batch, t=t, prompt_embeds=pe)
        loss = criterion(eps_pred.float(), noise.float())
        loss.backward()
        optimizer.step()
        running += loss.item()

    avg = running / max(1, (len(pairs) // batch_size))
    print(f"Epoch {epoch+1:03d} | Loss: {avg:.6f}")


Device: cuda


Keyword arguments {'trust_remote_code': True} are not expected by AudioLDM2Pipeline and will be ignored.
Loading pipeline components...: 100%|██████████| 11/11 [00:04<00:00,  2.45it/s]
Expected types for language_model: (<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>,), got <class 'transformers.models.gpt2.modeling_gpt2.GPT2Model'>.


subject 0001 data gathered
14 samples gathered
Latent shape: C=8, F=32, T=55


RuntimeError: mat1 and mat2 shapes cannot be multiplied (6x1024 and 768x1024)